# 🎵 Laxman Lofi AI Studio — Free GPU Runner V1.1

This runner provides **AI lyrics + ACE-Step music generation** from one API. It is designed to load the lyric LLM first, unload it, then load ACE-Step so a Colab T4 can reuse VRAM.

> ACE-Step v1 is currently listed as Apache 2.0 on Hugging Face. The default lyric model is `ministral/Ministral-3b-instruct`, also listed as Apache 2.0. Review current model terms and your publishing platform's rules before commercial distribution.

In [ ]:
!nvidia-smi
!git clone https://github.com/LaxmanNepal/lofi.git /content/lofi || true
!git clone https://github.com/ace-step/ACE-Step.git /content/ACE-Step || true
%cd /content/lofi
!pip -q install -e /content/ACE-Step
!pip -q install -r backend/requirements.txt nest-asyncio


In [ ]:
import os, sys, subprocess, time
os.environ['ACE_CHECKPOINT_DIR']='/content/ace-checkpoints'
os.environ['ACE_BF16']='true'
os.environ['LYRIC_MODEL_ID']='ministral/Ministral-3b-instruct'
os.environ['LYRIC_4BIT']='true'
# Optional API protection. Never commit the secret.
# os.environ['LAXMAN_LOFI_API_KEY']='change-this-secret'
subprocess.Popen([sys.executable,'-m','uvicorn','backend.server:app','--host','0.0.0.0','--port','8000'])
time.sleep(8)
!curl -s http://127.0.0.1:8000/health


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, time
log=open('/content/cloudflared.log','w')
subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8000'],stdout=log,stderr=subprocess.STDOUT)
time.sleep(6)
print(open('/content/cloudflared.log').read())


## Connect the web app

Copy the `https://....trycloudflare.com` URL from the previous cell. Open **Laxman Lofi → Settings (⚙)** and paste it into **ACE-Step API URL**.

Then you can click **✨ Create AI lyrics**. The server generates the lyric draft, releases the lyric model from VRAM, and then loads ACE-Step when you generate the song. Keep this Colab runtime alive while generating.